# Universidad Libre - Seccional Cali<br>Facultad de Ingeniería - Diplomado en Ciencia de Datos<br>(ↄ) Diego Fernando Marin, 2024

# 01_consolidar
Plantilla para el desarrollo del proyecto del diplomado de Ciencia de Datos, aplicando buenas prácticas.

---

Este cuaderno se enfoca en la integración de las distintas fuentes de datos en un formato cohesivo y estructurado. Aquí transformamos múltiples conjuntos de datos en una base unificada que servirá para los análisis posteriores.

**Propósito:** Crear una vista unificada y coherente de todos los datos recolectados, facilitando su posterior procesamiento y análisis.

**Tareas habituales:**
- Renombrar archivos
- Unión vertical de archivos complementarios (`union`)
- Combinar archivos (`joins`: inner, left, right, full outer)
- Estandarización inicial de formatos de columnas
- Verificación de consistencia en las uniones
- Validación de cardinalidad en las relaciones
- Gestión de duplicados producto de las uniones

In [ ]:
# ============================================================
# ZONA LANDING — Consolidar todos los CSV en un solo DataFrame
# ============================================================
from google.colab import drive
import pandas as pd
import os

drive.mount('/content/drive')

RAW_PATH     = "/content/drive/MyDrive/proyecto_oro/data/raw/"
LANDING_PATH = "/content/drive/MyDrive/proyecto_oro/data/landing/"

os.makedirs(LANDING_PATH, exist_ok=True)

# --- Leer y unir cada CSV por fecha ---
dfs = []

for archivo in os.listdir(RAW_PATH):
    if not archivo.endswith('.csv'):
        continue

    ruta = RAW_PATH + archivo
    nombre = archivo.replace('.csv', '')

    df = pd.read_csv(ruta, index_col=0, parse_dates=True)

    # Aplanar columnas multi-index si existen
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # Renombrar columnas con prefijo del activo
    df.columns = [f"{nombre}_{col}" for col in df.columns]

    dfs.append(df)
    print(f"✓ {archivo}: {df.shape[0]:,} filas × {df.shape[1]} columnas")

# --- Consolidar por fecha ---
print("\nConsolidando...")
df_landing = pd.concat(dfs, axis=1, join='outer')
df_landing = df_landing.sort_index()
df_landing = df_landing.ffill(limit=3)
df_landing = df_landing[df_landing.index.dayofweek < 5]  # solo días hábiles
df_landing.index.name = 'DATE'
df_landing = df_landing.reset_index()
df_landing['DATE'] = df_landing['DATE'].astype(str)

# --- Guardar en Landing ---
ruta_salida = LANDING_PATH + "consolidado_oro_dxy.csv"
df_landing.to_csv(ruta_salida, index=False)

print(f"\n✅ Landing listo")
print(f"📊 Shape: {df_landing.shape[0]:,} filas × {df_landing.shape[1]} columnas")
print(f"📅 Rango: {df_landing['DATE'].min()} → {df_landing['DATE'].max()}")
print(f"📁 Guardado en: {ruta_salida}")
print(f"\nColumnas:\n{list(df_landing.columns)}")
print(f"\nPrimeras filas:")
df_landing.head()

In [ ]:
import os
import pandas as pd

In [ ]:
cwd = os.getcwd() # current working directory
raw_dir = cwd + '/../data/raw/'
landing_dir = cwd + '/../data/landing/'

1. Recorriendo los diferentes directorios y sacando los nombres de los archivos:

In [ ]:
archivos = []
for ruta in os.listdir(raw_dir):
    if os.path.isdir(raw_dir + ruta):
        for archivo in os.listdir(raw_dir + ruta):
            # validar si es Excel?
            archivos.append(raw_dir + ruta + '/' + archivo)

In [ ]:
archivos

['/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2022/11_noviembre.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2022/12_diciembre.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2024/03_marzo.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2024/01_enero.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2024/02_febrero.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2023/03_marzo.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2023/11_noviembre.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2023/01_enero.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2023/07_julio.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2023/09_septiembre.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2023/04_abril.xlsx',
 '/Users/dfmarin/Public/

In [ ]:
df = pd.DataFrame()

for archivo in sorted(archivos):
    df_temp = pd.read_excel(archivo)
    df = pd.concat([df, df_temp], ignore_index=True)

In [ ]:
df

,fecha,codigo,venta
0,2022-11-01,1434,121672.710000
1,2022-11-02,1656,133636.640000
2,2022-11-03,1875,210932.180000
3,2022-11-04,1434,111660.390000
4,2022-11-05,1545,146417.560000
...,...,...,...
512,2024-03-27,1156,452331.135622
513,2024-03-28,1375,246045.083440
514,2024-03-29,934,425815.429745
515,2024-03-30,1156,320997.018300


2. Buscando todos los archivos con una condición, usando Glob:

In [ ]:
import glob

In [ ]:
archivos = glob.glob(raw_dir + '*/*.xlsx')

In [ ]:
archivos

['/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2022/11_noviembre.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2022/12_diciembre.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2024/03_marzo.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2024/01_enero.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2024/02_febrero.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2023/03_marzo.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2023/11_noviembre.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2023/01_enero.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2023/07_julio.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2023/09_septiembre.xlsx',
 '/Users/dfmarin/Public/Proyecto_Ciencia_de_Datos/src/../data/raw/2023/04_abril.xlsx',
 '/Users/dfmarin/Public/

In [ ]:
datos = []
for archivo in sorted(archivos):
    datos.append(pd.read_excel(archivo))

df = pd.concat(datos, ignore_index=True)

In [ ]:
df

,fecha,codigo,venta
0,2022-11-01,1434,121672.710000
1,2022-11-02,1656,133636.640000
2,2022-11-03,1875,210932.180000
3,2022-11-04,1434,111660.390000
4,2022-11-05,1545,146417.560000
...,...,...,...
512,2024-03-27,1156,452331.135622
513,2024-03-28,1375,246045.083440
514,2024-03-29,934,425815.429745
515,2024-03-30,1156,320997.018300


**Último paso**: Guardar los datos (union, joins) en un o más archivos, según sea necesario:

In [ ]:
df.to_csv(landing_dir + 'datos_consolidados.csv', index=False)